# Web Attack Detector — Model Evaluation

This notebook trains, evaluates, and visualizes all models.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

from data.dataset_generator import generate_dataset
from utils.feature_extractor import batch_extract
from models.classical_ml import AttackClassifier, AnomalyDetector

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print('Imports OK')

In [ ]:
# Generate dataset
df = generate_dataset(n_per_class=600)
print(df['label'].value_counts())
df.head()

In [ ]:
# Extract features
records = df.to_dict('records')
X = batch_extract(records)
y = df['label'].values
print('Feature matrix shape:', X.shape)

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Train classifier
ac = AttackClassifier()
ac.fit(X_train, y_train)
print('Classifier trained')

In [ ]:
# Classification report
labels_pred, _ = ac.predict(X_test)
print(classification_report(y_test, labels_pred))

In [ ]:
# Confusion matrix
label_names = ['normal', 'sqli', 'xss', 'cmdi', 'traversal']
cm = confusion_matrix(y_test, labels_pred, labels=label_names)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Attack Classifier — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Confidence distribution
_, confs = ac.predict(X_test)
plt.figure(figsize=(8, 4))
plt.hist(confs * 100, bins=30, color='steelblue', edgecolor='white')
plt.axvline(60, color='red', linestyle='--', label='QML gate (60%)')
plt.xlabel('Confidence %')
plt.ylabel('Count')
plt.title('Prediction Confidence Distribution')
plt.legend()
plt.tight_layout()
plt.show()
escalated = (confs < 0.60).sum()
print(f'Requests that would escalate to QML: {escalated}/{len(confs)} ({100*escalated/len(confs):.1f}%)')

In [ ]:
# Feature importance (from Random Forest inside ensemble)
rf_model = ac.ensemble.estimators_[0]  # RF is first
feature_names = [
    'url_len','params_len','full_len','url_entropy','params_entropy',
    'url_special','params_special','single_quotes','double_quotes',
    'dashes','equals','percent','sqli_kw','xss_kw','cmd_kw',
    'traversal_kw','enc_quote','enc_lt','enc_gt','enc_nl',
    'is_post','slash_count','q_count','amp_count','ext_len','suspicious_path'
]
importances = rf_model.feature_importances_
idx = importances.argsort()[::-1][:15]

plt.figure(figsize=(10, 5))
plt.bar(range(15), importances[idx], color='steelblue')
plt.xticks(range(15), [feature_names[i] for i in idx], rotation=45, ha='right')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()

In [ ]:
# Anomaly detector evaluation
X_normal = X_train[y_train == 'normal']
ad = AnomalyDetector(contamination=0.05)
ad.fit(X_normal)

# Check FPR on test normal traffic
X_test_normal = X_test[y_test == 'normal']
fp = ad.predict(X_test_normal).sum()
print(f'False positives on normal traffic: {fp}/{len(X_test_normal)} ({100*fp/len(X_test_normal):.1f}%)')

# Detection rate on attacks
X_test_attack = X_test[y_test != 'normal']
tp = ad.predict(X_test_attack).sum()
print(f'True positives on attack traffic: {tp}/{len(X_test_attack)} ({100*tp/len(X_test_attack):.1f}%)')